In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import sys
from pathlib import Path

# Adds the root_dir (parent of notebooks/) to sys.path
parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import pandas as pd 
import numpy as np  

from src.backtest_windows import generate_rolling_windows, split_data
from src.model_selector import SelectModel
from src.metrics import MetricsCalculator
from src.features import FeatureBuilder, TrainTestPrepare
from src.forecaster import Forecaster
from src.backtest_engine import BacktestEngine
from src.baselines import Baseline
from src.inventory_policy import InventoryPolicy

BASE_DIR = Path.cwd().parent


In [13]:
#load data 

train_df = pd.read_parquet(BASE_DIR/'data/processed/train_filtered_ca1.parquet')


In [14]:
# creat the backtest windows
windows = generate_rolling_windows(train_df,training_window=365,step_size=28,max_windows=10)

In [15]:
# 2. Build feature schema from a small sample
feat_builder = FeatureBuilder()

sample_df = train_df[train_df["item_id"] == train_df["item_id"].iloc[0]].tail(150)
_, full_features = feat_builder.build(sample_df)
full_features +=[c for c in feat_builder.build_sku_demand_profile(sample_df).columns if c !='item_id']
print("features used: ")
# full_features

features used: 


In [16]:
# initialize the classes
quantiles = [0.1,0.25,0.5,0.75,0.85,0.9,0.95]
cat_cols = ['item_id','dept_id','cat_id']
model = SelectModel(model='lgbm',quantiles=quantiles,use_log_transform=False,categorical_cols=cat_cols)

forecaster = Forecaster(feature_builder=feat_builder,original_features=full_features,
                        forecast_type='recursive')



In [ ]:
import re

from scripts.sku_diagnostics import per_sku_quantile_metrics

_QCOL = re.compile(r"q(\d{1,3})")

backtest = BacktestEngine(model_name='lgbm',forecast_type='direct')


def _quantile_levels(pred_df: pd.DataFrame) -> dict[str, float]:
    return {c: int(m.group(1)) / 100.0 for c in pred_df.columns
            if (m := _QCOL.fullmatch(c)) and 0 < int(m.group(1)) < 100}


def calculate_fill_rate(simulation_log: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Unit fill rate = fulfilled demand / demand = 1 - lost_sales / actual_sales.

    Returns per-SKU detail plus a volume-weighted (pooled) and unweighted
    (macro) summary. Zero-demand SKUs are excluded from the macro mean.
    """
    df = simulation_log.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['item_id'] = df['item_id'].astype(str)

    required = {'item_id', 'date', 'actual_sales', 'lost_sales'}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"simulation_log missing columns: {sorted(missing)}")

    sku = (
        df.groupby('item_id', observed=True)
        .agg(
            demand=('actual_sales', 'sum'),
            lost_sales=('lost_sales', 'sum'),
        )
        .reset_index()
    )
    sku['filled_units'] = sku['demand'] - sku['lost_sales']
    sku['fill_rate'] = np.where(
        sku['demand'] > 0,
        (sku['filled_units'] / sku['demand']).clip(0.0, 1.0),
        1.0,
    )

    demand = float(sku['demand'].sum())
    lost = float(sku['lost_sales'].sum())
    scored = sku['demand'] > 0

    overall = {
        'demand': demand,
        'lost_sales': lost,
        'filled_units': demand - lost,
        'fill_rate_pooled': 1.0 - lost / demand if demand > 0 else 1.0,
        'fill_rate_macro': float(sku.loc[scored, 'fill_rate'].mean()) if scored.any() else 1.0,
        'n_skus_scored': int(scored.sum()),
    }
    return sku, overall


def calc_inventory(train_df, test_df, preds_df, modelname, forecasttype, review_period=7, lead_time=4, coverage_days=4, policy='coverage'):
    risk_period = lead_time + review_period
    initial_inventory = (
        train_df
        .groupby('item_id', observed=True)['sales']
        .apply(lambda x: x.tail(28).mean())
    ) * risk_period

    # Merge actual sales into forecasts for error calculation
    preds_with_actuals = preds_df.merge(
        test_df[['item_id', 'date', 'sales']].copy(),
        on=['item_id', 'date'],
        how='left'
    )

    # Instantiate Inventory Policy using calibrated OOS errors & fresh initial stock
    inv_policy = InventoryPolicy(
        inventory=initial_inventory,
        forecasts=preds_with_actuals,
        lead_time=lead_time,
        review_period=review_period,
        coverage_days=coverage_days,
        daily_unit_holding_cost=0.2,
        stockout_cost=1,
        safety_stock_type=policy
    )

    sim_df = inv_policy.daily_simulation(
        actual_sales=test_df,
        forecasted_demand=preds_df,
        model_name=modelname,
        forecast_type=forecasttype,
        store_id='CA_1'
    )

    # Calculate holding, stockout, and total costs
    cost_summary = sim_df[['holding_cost', 'stockout_cost']].sum()
    cost_summary['total_cost'] = cost_summary.sum()

    return sim_df, cost_summary


def run_prob_backtest(
    df: pd.DataFrame,
    windows: list[dict],
    forecast_fn,
    feat_builder,
    horizon: int = 28,
    risk_period: int = 11,
    metrics: tuple = ('pinball', 'coverage', 'coverage_err'),
    max_windows: int | None = None,
    raise_on_error: bool = True,
    return_csl_details: bool = False,
    return_diagnostics: bool = False,
    policy='quantile',  # rmse, coverage, quantile
):
    """Run the probabilistic backtest.

    Returns a dict with two primary members:
        results     : inventory cost, realised fill rate and realised CSL only
        calibration : per-quantile pinball / coverage / coverage error

    When return_csl_details=True, the dict also carries:
        cycle_log   : per-SKU review-cycle CSL detail
        sku_csl     : per-SKU realised CSL joined with realised fill rate
    """
    rows = []
    calibration_rows = []
    all_cycle_logs = []
    all_sku_csl = []
    diagnostic_rows = []
    aligned_rows = []

    for wid, w in enumerate(windows[:max_windows] if max_windows else windows):
        try:
            print(f'window {wid}:')
            ts, te = pd.Timestamp(w['train_start']), pd.Timestamp(w['train_end'])
            ee = pd.Timestamp(w['test_end'])
            slice_ = feat_builder.add_price_features(
                df[(df['date'] >= ts) & (df['date'] <= ee)])
            train_wnd, eval_wnd = split_data(slice_, ts, te, forecast_horizon=horizon)

            # evaluate baselines with inventory coverage policy
            base_preds, _ = backtest._baselines_metrics(
                train_df=train_wnd, eval_df=eval_wnd)

            profile = feat_builder.build_sku_demand_profile(train_wnd)
            train_wnd = train_wnd.merge(profile, on='item_id', how='left')
            eval_wnd = eval_wnd.merge(profile, on='item_id', how='left')
            train_wnd, _ = feat_builder.build(df=train_wnd)

            pred = forecast_fn(train_wnd, eval_wnd, type='direct')
            if 'sales_pred' not in pred and 'point' in pred:
                pred = pred.rename(columns={'point': 'sales_pred'})

            base_preds_items = base_preds.items() if isinstance(base_preds, dict) else base_preds
            model_preds = dict(base_preds_items)
            model_preds['ml-direct'] = pred

            for name, preds in model_preds.items():
                is_ml = name == 'ml-direct'
                inventory, cost_series = calc_inventory(
                    train_df=train_wnd,
                    test_df=eval_wnd,
                    preds_df=preds,
                    modelname=name,
                    forecasttype=name,
                    policy=policy if is_ml else 'coverage',
                )

                # results: inventory cost only
                rows.append({
                    'window_id': w.get('window_id', wid),
                    'metric': 'inventory_cost',
                    'model': name,
                    **cost_series.to_dict(),
                })

                # results: realised fill rate
                fill_sku, fill_overall = calculate_fill_rate(inventory)
                rows.append({
                    'window_id': w.get('window_id', wid),
                    'metric': 'fill_rate_pooled',
                    'model': name,
                    'value': fill_overall['fill_rate_pooled'],
                })
                rows.append({
                    'window_id': w.get('window_id', wid),
                    'metric': 'fill_rate_macro',
                    'model': name,
                    'value': fill_overall['fill_rate_macro'],
                })

                # results: realised CSL
                try:
                    cycle_log, sku_csl, overall_csl = calculate_realized_csl(
                        inventory,
                        review_period=7,
                        protection_period=risk_period,
                    )
                except ValueError as exc:
                    print(f'  skipping CSL for {name}: {exc}')
                    cycle_log = sku_csl = overall_csl = None

                if overall_csl is not None:
                    rows.append({
                        'window_id': w.get('window_id', wid),
                        'metric': 'realized_csl_pooled',
                        'model': name,
                        'value': overall_csl['pooled_csl'],
                    })
                    rows.append({
                        'window_id': w.get('window_id', wid),
                        'metric': 'realized_csl_macro',
                        'model': name,
                        'value': overall_csl['macro_csl'],
                    })

                    cycle_log['window_id'] = w.get('window_id', wid)
                    cycle_log['model'] = name
                    sku_csl['window_id'] = w.get('window_id', wid)
                    sku_csl['model'] = name
                    sku_csl = sku_csl.merge(
                        fill_sku[['item_id', 'demand', 'lost_sales', 'fill_rate']],
                        on='item_id',
                        how='left',
                    )
                    all_cycle_logs.append(cycle_log)
                    all_sku_csl.append(sku_csl)

                # separate calibration table: quantile metrics only
                q_levels = _quantile_levels(preds)
                calc = MetricsCalculator(
                    train_wnd, eval_wnd, preds, risk_period=risk_period)
                if q_levels:
                    pinball = calc.pinball_loss(q_levels)
                    coverage = calc.coverage_level(q_levels)
                    for col, level in q_levels.items():
                        row = {
                            'window_id': w.get('window_id', wid),
                            'model': name,
                            'quantile': col,
                            'level': level,
                        }
                        if 'pinball' in metrics:
                            row['pinball'] = pinball[col]
                        if 'coverage' in metrics:
                            row['coverage'] = coverage[col]
                        if 'coverage_err' in metrics:
                            row['coverage_err'] = coverage[col] - level
                        calibration_rows.append(row)

                if return_diagnostics:
                    policy_label = policy if is_ml else 'coverage'
                    aligned = calc.aligned.copy()
                    aligned['item_id'] = aligned['item_id'].astype(str)
                    aligned['window_id'] = w.get('window_id', wid)
                    aligned['model'] = name
                    aligned['policy_label'] = policy_label
                    aligned['horizon'] = (
                        pd.to_datetime(aligned['date']) - te).dt.days

                    keep = ['window_id', 'model', 'policy_label', 'item_id',
                            'date', 'horizon', 'sales', 'sales_pred']
                    keep += list(q_levels)
                    aligned_rows.append(
                        aligned[[c for c in keep if c in aligned.columns]])

                    sku_diag = per_sku_quantile_metrics(aligned, q_levels)
                    sku_diag['window_id'] = w.get('window_id', wid)
                    sku_diag['model'] = name
                    sku_diag['policy_label'] = policy_label

                    inv_sku = (
                        inventory.groupby('item_id', observed=True)
                        .agg(
                            holding_cost=('holding_cost', 'sum'),
                            stockout_cost=('stockout_cost', 'sum'),
                            on_hand_mean=('on_hand', 'mean'),
                            inventory_days=('date', 'nunique'),
                        )
                        .reset_index()
                    )
                    inv_sku['item_id'] = inv_sku['item_id'].astype(str)
                    sku_diag = sku_diag.merge(inv_sku, on='item_id', how='left')

                    if fill_sku is not None and not fill_sku.empty:
                        sku_diag = sku_diag.merge(
                            fill_sku[['item_id', 'lost_sales', 'filled_units', 'fill_rate']],
                            on='item_id',
                            how='left',
                        )

                    if sku_csl is not None and not sku_csl.empty:
                        sku_diag = sku_diag.merge(
                            sku_csl[['item_id', 'total_cycles', 'successful_cycles',
                                     'realized_csl', 'total_lost_sales']],
                            on='item_id',
                            how='left',
                        )

                    diagnostic_rows.append(sku_diag)
        except Exception as exc:
            print(f'Window {wid} failed: {exc}')
            if raise_on_error:
                raise

    out = {
        'results': pd.DataFrame(rows),
        'calibration': pd.DataFrame(calibration_rows),
    }

    if return_csl_details:
        out['cycle_log'] = (
            pd.concat(all_cycle_logs, ignore_index=True)
            if all_cycle_logs else pd.DataFrame()
        )
        out['sku_csl'] = (
            pd.concat(all_sku_csl, ignore_index=True)
            if all_sku_csl else pd.DataFrame()
        )

    if return_diagnostics:
        out['diagnostics'] = (
            pd.concat(diagnostic_rows, ignore_index=True)
            if diagnostic_rows else pd.DataFrame()
        )
        out['aligned'] = (
            pd.concat(aligned_rows, ignore_index=True)
            if aligned_rows else pd.DataFrame()
        )

    return out

TARGET_KNOWN = ['snap_CA', 'is_event', 'is_event_in_7_days', 'day_of_week', 'month',
                'sell_price', 'is_sporting_event', 'is_cultural_event',
                'is_national_event', 'is_religious_event', 'day_of_month', 'week_of_year']

def _direct_frames(train_df, test_df, horizon):
    tgt = [c for c in TARGET_KNOWN if c in train_df.columns]
    tr, cols = TrainTestPrepare.build_direct_train_frame(
        known_df=train_df, target_known_cols=tgt,
        original_feature_cols=full_features, horizon=horizon)
    te = TrainTestPrepare.build_direct_test_frame(
        future_df=test_df, history_df=train_df,
        original_feature_cols=full_features, target_known_cols=tgt, max_horizon=horizon)
    return tr, te, cols

def _assemble(test_df, sales_pred, quantiles):
    out = {'item_id': test_df['item_id'].values, 'date': test_df['target_date'].values,
           'sales_pred': np.asarray(sales_pred, dtype=float)}
    out.update(quantiles)
    return pd.DataFrame(out)

def _sorted_quantiles(qs):
    names = sorted(qs, key=lambda k: int(k[1:]))
    Q = np.column_stack([qs[k] for k in names])
    Q.sort(axis=1)                                   # enforce monotonicity
    return {k: Q[:, j] for j, k in enumerate(names)}

def plain_direct(train_df, test_df, type='direct', horizon=28):
    '''Baseline: quantile models fitted on ALL rows, no classifier.'''
    tr, te, cols = _direct_frames(train_df, test_df, horizon)
    y = tr['target_sales'].to_numpy()
    for m in models.values():
        m.fit(tr[cols], y)
    X = te[cols]
    qs = {k: np.clip(m.predict(X), 0, None) for k, m in models.items() if k != 'point'}
    return _assemble(te, np.clip(models['point'].predict(X), 0, None), _sorted_quantiles(qs))

def two_stage_hurdle(train_df, test_df, type='direct', horizon=28):
    '''Classifier for sell/no-sell + quantiles on positive rows, recombined into
    all-day quantiles: 0 inside the zero mass, else the positive model at level
    beta = (alpha - zero_share) / p, interpolated on the fitted quantile grid.'''
    tr, te, cols = _direct_frames(train_df, test_df, horizon)
    y = tr['target_sales'].to_numpy()
    pos = y > 0
    lgb_binary.fit(tr[cols], pos.astype(int))
    for m in models.values():
        m.fit(tr.loc[pos, cols], y[pos])

    X = te[cols]
    p = lgb_binary.predict_proba(X)[:, 1]
    z = 1.0 - p
    names = sorted([k for k in models if k != 'point'], key=lambda k: int(k[1:]))
    lv = np.array([int(k[1:]) / 100 for k in names])
    Q = np.column_stack([np.clip(models[k].predict(X), 0, None) for k in names])

    grid_lv = np.r_[0.0, lv]                          # positive-level grid
    grid_q = np.column_stack([np.zeros(len(X)), Q])
    rows = np.arange(len(X))
    qs = {}
    for k, alpha in zip(names, lv):
        beta = np.clip((alpha - z) / np.maximum(p, 1e-9), 0.0, 1.0)  # level inside positive part
        i = np.clip(np.searchsorted(grid_lv, beta, side='right') - 1, 0, len(grid_lv) - 2)
        w = np.clip((beta - grid_lv[i]) / (grid_lv[i + 1] - grid_lv[i]), 0.0, 1.0)
        q = grid_q[rows, i] * (1 - w) + grid_q[rows, i + 1] * w
        qs[k] = np.where(alpha <= z, 0.0, q)          # inside zero mass -> 0

    qs = _sorted_quantiles(qs)
    return _assemble(te, p * np.clip(models['point'].predict(X), 0, None), qs)


def simulate_lead_time_demand(quantile_preds:np.ndarray,quantile_levels:list, risk_period = 11,n_simulations=5000,target_csl=83.33):
    horizon_days, n_quantiles = quantile_preds.shape
    assert len(quantile_levels) == n_quantiles, "Quantile levels mismatch prediction shape."
    
    # 1. Generate uniform random variables for all simulations and lead time days
    # Shape: (n_simulations, lead_time)
    u_draws = np.random.uniform(0, 1, size=(n_simulations, risk_period))
    
    simulated_ltd = np.zeros(n_simulations)

    for t in range(risk_period):
        day_quantiles = quantile_preds[t,:]
        q_min, q_max = quantile_levels[0],quantile_levels[-1]

        simulated_daily_demand = np.interp(u_draws[:,t],xp=quantile_levels,fp=day_quantiles)
        simulated_daily_demand = np.clip(simulated_daily_demand,0,None)
        simulated_ltd +=simulated_daily_demand
    # 3. Derive inventory target from the empirical Lead Time Demand distribution
    inventory_target = np.quantile(simulated_ltd, q=target_csl)
    expected_ltd = np.mean(simulated_ltd)
    safety_stock = inventory_target - np.quantile(simulated_ltd, q=0.5) # median-based buffer

    return {
        "inventory_target": inventory_target,
        "expected_ltd": expected_ltd,
        "safety_stock": safety_stock,
        "ltd_distribution": simulated_ltd
    }


def calculate_realized_csl(
    simulation_log: pd.DataFrame,
    review_period: int = 7,
    protection_period: int = 11,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """
    Calculate realised protection-period CSL.

    A SKU-cycle is successful if it experiences zero lost sales
    during the complete protection period following a review date.

    Returns:
        cycle_log: One row per SKU and review cycle.
        sku_summary: CSL and cycle counts for each SKU.
        overall_summary: Pooled and macro CSL.
    """

    df = simulation_log.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["item_id"] = df["item_id"].astype(str)

    dates = pd.DatetimeIndex(sorted(df["date"].unique()))

    if len(dates) == 0:
        raise ValueError("Simulation log is empty.")

    # Match the review schedule in daily_simulation:
    # first simulation date, then every review_period days.
    review_dates = dates[::review_period]

    available_dates = set(dates)
    cycle_records = []

    for review_date in review_dates:

        window_dates = pd.date_range(
            start=review_date,
            periods=protection_period,
            freq="D",
        )

        # Only evaluate complete protection periods.
        if not all(d in available_dates for d in window_dates):
            continue

        window = df[df["date"].isin(window_dates)]

        # Total lost sales and number of stockout days per SKU.
        cycle = (
            window.groupby("item_id", observed=True)
            .agg(
                lost_sales=("lost_sales", "sum"),
                stockout_days=(
                    "lost_sales",
                    lambda x: (x > 0).sum()
                ),
            )
            .reset_index()
        )

        cycle["review_date"] = review_date

        # A cycle fails if even one unit of demand is lost.
        cycle["cycle_success"] = cycle["lost_sales"].eq(0)

        cycle_records.append(cycle)

    if not cycle_records:
        raise ValueError(
            "No complete protection periods found. "
            "The simulation needs more days."
        )

    cycle_log = pd.concat(cycle_records, ignore_index=True)

    # Per-SKU realised CSL.
    sku_summary = (
        cycle_log.groupby("item_id", observed=True)
        .agg(
            total_cycles=("cycle_success", "size"),
            successful_cycles=("cycle_success", "sum"),
            realized_csl=("cycle_success", "mean"),
            total_lost_sales=("lost_sales", "sum"),
        )
        .reset_index()
    )

    # Two ways of aggregating across SKUs.
    pooled_csl = cycle_log["cycle_success"].mean()
    macro_csl = sku_summary["realized_csl"].mean()

    overall_summary = {
        "total_sku_cycles": len(cycle_log),
        "successful_sku_cycles": int(
            cycle_log["cycle_success"].sum()
        ),
        "pooled_csl": pooled_csl,
        "macro_csl": macro_csl,
        "number_of_skus": sku_summary["item_id"].nunique(),
    }

    return cycle_log, sku_summary, overall_summary


In [ ]:
base_run = dict(
    df=train_df,
    windows=windows[:1],
    forecast_fn=plain_direct,
    feat_builder=feat_builder,
    return_csl_details=True,
    return_diagnostics=True,
)

out = run_prob_backtest(**base_run, policy='quantile')
coverage_out = run_prob_backtest(**base_run, policy='coverage')

results = out['results']
calibration = out['calibration']
cycle_log = out['cycle_log']
sku_csl = out['sku_csl']
diagnostics = out['diagnostics']
aligned = out['aligned']


In [ ]:
from pathlib import Path
import json
import pandas as pd

OUT = Path('../results/diagnostics/window_0')
OUT.mkdir(parents=True, exist_ok=True)


def _ml_rows(frame):
    if frame is None or frame.empty or 'model' not in frame.columns:
        return frame
    return frame.loc[frame['model'].eq('ml-direct')].copy()


diag_all = pd.concat(
    [diagnostics, _ml_rows(coverage_out['diagnostics'])],
    ignore_index=True,
)
diag_all = diag_all.drop_duplicates(
    subset=['window_id', 'model', 'policy_label', 'item_id']
)

prob_results = out['results'].copy()
prob_results['policy_label'] = np.where(
    prob_results['model'].eq('ml-direct'), 'quantile', 'coverage')
cov_results = _ml_rows(coverage_out['results']).copy()
cov_results['policy_label'] = 'coverage'

prob_sku_csl = out['sku_csl'].copy()
prob_sku_csl['policy_label'] = np.where(
    prob_sku_csl['model'].eq('ml-direct'), 'quantile', 'coverage')
cov_sku_csl = _ml_rows(coverage_out['sku_csl']).copy()
cov_sku_csl['policy_label'] = 'coverage'

prob_cycle_log = out['cycle_log'].copy()
prob_cycle_log['policy_label'] = np.where(
    prob_cycle_log['model'].eq('ml-direct'), 'quantile', 'coverage')
cov_cycle_log = _ml_rows(coverage_out['cycle_log']).copy()
cov_cycle_log['policy_label'] = 'coverage'

results_all = pd.concat([prob_results, cov_results], ignore_index=True)
sku_csl_all = pd.concat([prob_sku_csl, cov_sku_csl], ignore_index=True)
cycle_log_all = pd.concat([prob_cycle_log, cov_cycle_log], ignore_index=True)
aligned_all = pd.concat(
    [out['aligned'], _ml_rows(coverage_out['aligned'])],
    ignore_index=True,
)

w0 = windows[0]
ts, te = pd.Timestamp(w0['train_start']), pd.Timestamp(w0['train_end'])
train_window = train_df[(train_df['date'] >= ts) & (train_df['date'] <= te)]
profile = feat_builder.build_sku_demand_profile(train_window)

meta_columns = [c for c in ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
                if c in train_window.columns]
sku_meta = train_window[meta_columns].drop_duplicates('item_id').reset_index(drop=True)

diag_all.to_parquet(OUT / 'diagnostics.parquet', index=False)
aligned_all.to_parquet(OUT / 'aligned.parquet', index=False)
results_all.to_parquet(OUT / 'results.parquet', index=False)
out['calibration'].to_parquet(OUT / 'calibration.parquet', index=False)
sku_csl_all.to_parquet(OUT / 'sku_csl.parquet', index=False)
cycle_log_all.to_parquet(OUT / 'cycle_log.parquet', index=False)
profile.to_parquet(OUT / 'profile.parquet', index=False)
sku_meta.to_parquet(OUT / 'sku_meta.parquet', index=False)

metadata = {
    'window': w0,
    'generated_at': pd.Timestamp.now().isoformat(),
    'models': sorted(diag_all['model'].unique().tolist()),
    'policies': sorted(diag_all['policy_label'].unique().tolist()),
    'n_skus': int(diag_all['item_id'].nunique()),
    'service_quantile': 0.8333333333,
}
(OUT / 'metadata.json').write_text(json.dumps(metadata, indent=2, default=str))
print(f'wrote diagnostics bundle to {OUT.resolve()}')


In [22]:
# # Local inventory engine for probabilistic experiments.
# # Run this cell after the cell that defines run_prob_backtest.
# # Re-run it if you re-run that earlier cell; this overrides calc_inventory.

# import re
# import numpy as np
# import pandas as pd

# LEAD_TIME = 4
# REVIEW_PERIOD = 7
# COVERAGE_DAYS = 4
# RISK_PERIOD = LEAD_TIME + REVIEW_PERIOD
# HOLDING_COST = 0.2
# STOCKOUT_COST = 1.0
# TARGET_SERVICE_LEVEL = 1.0 / (1.0 + HOLDING_COST / STOCKOUT_COST)
# N_SIMS = 5000
# SEED = 42


# def _quantile_columns(df):
#     cols = [c for c in df.columns if re.fullmatch(r"q\d{1,3}", str(c))]
#     return sorted(cols, key=lambda c: int(c[1:]))




# def _simulate_demand_paths(
#     item_df, q_cols, n_sims=N_SIMS, seed=SEED, u_draws=None
# ):
#     """Return simulated daily demand paths and their dates for one SKU."""
#     item_df = item_df.sort_values("date")
#     q_levels = np.array([int(c[1:]) / 100.0 for c in q_cols])
#     q_preds = np.nan_to_num(
#         item_df[q_cols].to_numpy(dtype=float)[:RISK_PERIOD], nan=0.0
#     )
#     q_preds = np.clip(np.maximum.accumulate(q_preds, axis=1), 0.0, None)

#     horizon = q_preds.shape[0]
#     if u_draws is not None:
#         u_draws = np.asarray(u_draws)
#         n_sims = u_draws.shape[0]
#         horizon = min(horizon, u_draws.shape[1])
#         q_preds = q_preds[:horizon]

#     dates = item_df["date"].iloc[:horizon].to_numpy()
#     if horizon == 0:
#         return np.zeros((n_sims, 0)), dates

#     if u_draws is None:
#         rng = np.random.default_rng(seed)
#         u_draws = rng.uniform(0.0, 1.0, size=(n_sims, horizon))
#     else:
#         u_draws = u_draws[:, :horizon]

#     paths = np.zeros((n_sims, horizon))
#     for t in range(horizon):
#         paths[:, t] = np.clip(
#             np.interp(u_draws[:, t], q_levels, q_preds[t]), 0.0, None
#         )
#     return paths, dates


# def simulate_ltd(item_df, q_cols, n_sims=N_SIMS, seed=SEED, u_draws=None):
#     """Simulate lead-time demand for one SKU from its quantile forecasts."""
#     paths, _ = _simulate_demand_paths(
#         item_df, q_cols, n_sims=n_sims, seed=seed, u_draws=u_draws
#     )
#     return paths.sum(axis=1)


# def simulate_daily_demand_paths(item_df, q_cols=None, n_paths=100, seed=SEED):
#     """Simulated daily demand paths for one SKU as a wide DataFrame.

#     Rows are dates, columns are paths. Use ``paths.sum(axis=0)`` for the
#     corresponding lead-time-demand samples.
#     """
#     q_cols = q_cols or _quantile_columns(item_df)
#     if not q_cols:
#         raise ValueError("No quantile columns found (expected q10, q25, ...).")
#     paths, dates = _simulate_demand_paths(
#         item_df, q_cols, n_sims=n_paths, seed=seed
#     )
#     return pd.DataFrame(
#         paths.T,
#         index=pd.Index(dates, name="date"),
#         columns=[f"path_{i}" for i in range(n_paths)],
#     )


# def get_simulated_paths(forecasts, item_id, n_paths=100, seed=SEED):
#     """Demand paths for a selected item; useful for fan charts in posts."""
#     item_df = forecasts[forecasts["item_id"].astype(str) == str(item_id)].copy()
#     if item_df.empty:
#         raise ValueError(f"No forecasts found for item_id={item_id!r}.")
#     return simulate_daily_demand_paths(item_df, n_paths=n_paths, seed=seed)


# def order_up_to_quantile(forecast_slice):
#     """Quantile policy: S = target service quantile of simulated lead-time demand."""
#     fc = forecast_slice.copy()
#     fc["date"] = pd.to_datetime(fc["date"])
#     fc["item_id"] = fc["item_id"].astype(str)

#     q_cols = _quantile_columns(fc)
#     if not q_cols:
#         raise ValueError("No quantile columns found (expected q10, q25, ...).")

#     start = fc["date"].min()
#     risk_end = start + pd.Timedelta(days=RISK_PERIOD - 1)
#     risk = fc[fc["date"] <= risk_end]

#     risk_demand = risk.groupby("item_id", observed=True)["sales_pred"].sum()
#     safety_stock = {}
#     for item_id, item_df in risk.groupby("item_id", observed=True):
#         ltd = simulate_ltd(item_df, q_cols)
#         target = np.quantile(ltd, TARGET_SERVICE_LEVEL)
#         safety_stock[item_id] = max(0.0, target - risk_demand.loc[item_id])

#     safety_stock = pd.Series(safety_stock).reindex(risk_demand.index, fill_value=0.0)
#     return risk_demand.add(safety_stock, fill_value=0.0), safety_stock


# def order_up_to_coverage(forecast_slice):
#     """Coverage policy: risk-period demand plus the next COVERAGE_DAYS of forecast."""
#     fc = forecast_slice.copy()
#     fc["date"] = pd.to_datetime(fc["date"])
#     fc["item_id"] = fc["item_id"].astype(str)

#     start = fc["date"].min()
#     risk_end = start + pd.Timedelta(days=RISK_PERIOD - 1)
#     coverage_end = risk_end + pd.Timedelta(days=COVERAGE_DAYS)

#     risk = fc[fc["date"] <= risk_end]
#     coverage = fc[(fc["date"] > risk_end) & (fc["date"] <= coverage_end)]

#     risk_demand = risk.groupby("item_id", observed=True)["sales_pred"].sum()
#     coverage_demand = coverage.groupby("item_id", observed=True)["sales_pred"].sum()
#     coverage_demand = coverage_demand.reindex(risk_demand.index, fill_value=0.0)

#     return risk_demand.add(coverage_demand, fill_value=0.0), coverage_demand


# def simulate_inventory(actual_sales, forecasts, initial_inventory, policy="quantile"):
#     """Daily periodic-review simulation. policy is 'quantile', 'coverage', or a callable."""
#     order_fn = policy if callable(policy) else {
#         "quantile": order_up_to_quantile,
#         "coverage": order_up_to_coverage,
#     }[policy]

#     actual = actual_sales.copy()
#     fc = forecasts.copy()
#     for frame in (actual, fc):
#         frame["date"] = pd.to_datetime(frame["date"])
#         frame["item_id"] = frame["item_id"].astype(str)

#     on_hand = initial_inventory.copy()
#     on_hand.index = on_hand.index.astype(str)
#     items = on_hand.index

#     pipeline = []
#     logs = []
#     last_safety_stock = pd.Series(0.0, index=items)

#     for idx, day in enumerate(sorted(fc["date"].unique())):
#         order_qty = pd.Series(0.0, index=items)
#         order_up_to = pd.Series(0.0, index=items)

#         if idx % REVIEW_PERIOD == 0:
#             future = fc[fc["date"] >= day]
#             on_order = pd.Series(0.0, index=items)
#             for arrival, qty in pipeline:
#                 if arrival >= day:
#                     on_order = on_order.add(qty.reindex(items, fill_value=0.0), fill_value=0.0)

#             target, safety_stock = order_fn(future)
#             target = target.reindex(items, fill_value=0.0)
#             safety_stock = safety_stock.reindex(items, fill_value=0.0)

#             order_qty = (target - on_hand - on_order).clip(lower=0.0)
#             order_up_to = target
#             last_safety_stock = safety_stock
#             pipeline.append((day + pd.Timedelta(days=LEAD_TIME), order_qty))

#         arriving = pd.Series(0.0, index=items)
#         for arrival, qty in pipeline:
#             if arrival == day:
#                 arriving = arriving.add(qty.reindex(items, fill_value=0.0), fill_value=0.0)

#         sales_today = (
#             actual.loc[actual["date"].eq(day)]
#             .groupby("item_id", observed=True)["sales"]
#             .sum()
#             .reindex(items, fill_value=0.0)
#         )

#         on_hand = on_hand.add(arriving, fill_value=0.0)
#         fulfilled = np.minimum(on_hand, sales_today)
#         lost_sales = (sales_today - fulfilled).clip(lower=0.0)
#         on_hand = (on_hand - fulfilled).clip(lower=0.0)

#         logs.append(pd.DataFrame({
#             "date": day,
#             "item_id": items,
#             "on_hand": on_hand.values,
#             "arriving_qty": arriving.values,
#             "actual_sales": sales_today.values,
#             "safety_stock": last_safety_stock.values,
#             "lost_sales": lost_sales.values,
#             "order_qty": order_qty.values,
#             "order_up_to": order_up_to.values,
#             "holding_cost": (on_hand * HOLDING_COST).values,
#             "stockout_cost": (lost_sales * STOCKOUT_COST).values,
#         }))

#     return pd.concat(logs, ignore_index=True)


# def get_inventory_path(sim_df, item_id):
#     """Daily inventory simulation for a selected item."""
#     return (
#         sim_df[sim_df["item_id"].astype(str) == str(item_id)]
#         .sort_values("date")
#         .reset_index(drop=True)
#     )


# def calc_inventory(train_df, test_df, preds_df, policy="quantile", **kwargs):
#     """Simple wrapper used by the notebook. Extra kwargs are ignored for compatibility."""
#     initial_inventory = (
#         train_df.groupby("item_id", observed=True)["sales"]
#         .apply(lambda x: x.tail(28).mean())
#     ) * RISK_PERIOD

#     sim_df = simulate_inventory(test_df, preds_df, initial_inventory, policy)
#     cost_summary = sim_df[["holding_cost", "stockout_cost"]].sum()
#     cost_summary["total_cost"] = cost_summary.sum()
#     return sim_df, cost_summary


## Two stage hurdle approach:
1. classification, 2. prediction 

In [19]:
from lightgbm import LGBMClassifier, LGBMRegressor

lgb_binary = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=8,
    num_leaves=40,
    random_state=42,
    verbose=-1
)

params = dict(
            tweedie_variance_power=1.2,
            n_estimators=500,
            learning_rate=0.03,
            num_leaves=15,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
        )

lgb_regressor = LGBMRegressor(objective='tweedie',**params)

In [20]:
quant_models = {}     
params = dict(
            tweedie_variance_power=1.2,
            n_estimators=500,
            learning_rate=0.03,
            num_leaves=15,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
        )
quant_models = {f'q{int(q*100)}':LGBMRegressor(objective='quantile',alpha=q,**params)
                for q in quantiles}

models = {'point':lgb_regressor,**quant_models}

# models

In [25]:
results

,window_id,metric,model,holding_cost,stockout_cost,total_cost,value
0,window_0,inventory_cost,seasonal_naive,18580.050000,1604.857143,20184.907143,NaN
1,window_0,inventory_cost,simple_moving_average,17440.962222,1567.000794,19007.963016,NaN
2,window_0,inventory_cost,seasonal_moving_average,18147.627624,1536.017871,19683.645495,NaN
3,window_0,inventory_cost,ml-direct,15419.827918,1323.113006,16742.940925,NaN
4,window_0,realized_csl_pooled,ml-direct,NaN,NaN,NaN,0.662222
5,window_0,realized_csl_macro,ml-direct,NaN,NaN,NaN,0.662222


In [29]:
sku_csl["realized_csl"].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
)

count    300.000000
mean       0.662222
std        0.375755
min        0.000000
10%        0.000000
25%        0.333333
50%        1.000000
75%        1.000000
90%        1.000000
max        1.000000
Name: realized_csl, dtype: float64

In [35]:
sku_csl["realized_csl"].value_counts().sort_index()

realized_csl
0.000000     37
0.333333     81
0.666667     31
1.000000    151
Name: count, dtype: int64

In [36]:
sku_csl["csl_group"] = pd.cut(
    sku_csl["realized_csl"],
    bins=[-0.01, 0, 0.34, 0.67, 1.01],
    labels=["0%", "1–33%", "34–66%", "67–100%"]
)

sku_csl["csl_group"].value_counts().sort_index()

csl_group
0%          37
1–33%       81
34–66%      31
67–100%    151
Name: count, dtype: int64

In [ ]:
lost = sku_csl.sort_values(
    "total_lost_sales",
    ascending=False
).copy()

total_lost = lost["total_lost_sales"].sum()

lost["cumulative_share"] = (
    lost["total_lost_sales"].cumsum() / total_lost
)

for pct in [0.10, 0.20, 0.30]:
    n = int(np.ceil(len(lost) * pct))
    share = lost.iloc[:n]["total_lost_sales"].sum() / total_lost

    print(
        f"Top {pct:.0%} of SKUs account for "
        f"{share:.1%} of lost sales"
    )

Top 10% of SKUs account for 68.5% of lost sales
Top 20% of SKUs account for 86.0% of lost sales
Top 30% of SKUs account for 94.8% of lost sales


In [ ]:
runs = {'plain': plain_direct}
parts = []
for name, fn in runs.items():
    out = run_prob_backtest(train_df, windows[:1], forecast_fn=fn, feat_builder=FeatureBuilder())
    r = out['results']
    if 'model' in r.columns:
        r = r.drop(columns=['model'])
    r.insert(0, 'model', name)
    parts.append(r)
cmp_df = pd.concat(parts, ignore_index=True)
cmp_df


In [12]:
# Experiment adapted to the local inventory engine above.
# The previous experiment cell is kept intact; this is a separate run that
# also exposes simulated paths for a selected item.

from IPython.display import display

assert globals().get("LOCAL_INVENTORY_ENGINE", False), (
    "Run the local inventory engine cell above before this experiment."
)

WINDOW = windows[0]
SELECTED_MODEL = "hurdle"   # "plain" or "hurdle"
SELECTED_ITEM = None        # None -> generate paths for every item
N_PATHS = 100

horizon = 28
ts = pd.Timestamp(WINDOW["train_start"])
te = pd.Timestamp(WINDOW["train_end"])
ee = pd.Timestamp(WINDOW["test_end"])

slice_ = feat_builder.add_price_features(
    train_df[(train_df["date"] >= ts) & (train_df["date"] <= ee)]
)
train_wnd, eval_wnd = split_data(slice_, ts, te, forecast_horizon=horizon)

if SELECTED_ITEM is not None:
    SELECTED_ITEM = str(SELECTED_ITEM)

# Baseline reference costs using the local coverage policy.
base_preds, _ = backtest._baselines_metrics(
    train_df=train_wnd, eval_df=eval_wnd
)
cost_rows = []
for name, preds in base_preds.items():
    sim, cost = calc_inventory(
        train_wnd, eval_wnd, preds, policy="coverage"
    )
    cost_rows.append({"model": f"baseline:{name}", **cost.to_dict()})

# ML runs using the local quantile policy.
profile = feat_builder.build_sku_demand_profile(train_wnd)
train_wnd = train_wnd.merge(profile, on="item_id", how="left")
eval_wnd = eval_wnd.merge(profile, on="item_id", how="left")
train_wnd, _ = feat_builder.build(df=train_wnd)

runs = {"plain": plain_direct, "hurdle": two_stage_hurdle}
preds_by_model = {}
sims_by_model = {}

for name, fn in runs.items():
    pred = fn(train_wnd, eval_wnd, type="direct")
    if "sales_pred" not in pred and "point" in pred:
        pred = pred.rename(columns={"point": "sales_pred"})
    preds_by_model[name] = pred

    sim, cost = calc_inventory(
        train_wnd, eval_wnd, pred, policy="quantile"
    )
    sims_by_model[name] = sim
    cost_rows.append({"model": name, **cost.to_dict()})

cost_df = (
    pd.DataFrame(cost_rows)
    .sort_values("total_cost")
    .reset_index(drop=True)
)

selected_pred = preds_by_model[SELECTED_MODEL]
selected_sim = sims_by_model[SELECTED_MODEL]

if SELECTED_ITEM is None:
    item_ids = selected_pred["item_id"].astype(str).unique().tolist()
else:
    item_ids = [str(SELECTED_ITEM)]

paths_by_item = {}
ltd_by_item = {}
inv_paths_by_item = {}

for item_id in item_ids:
    item_paths = get_simulated_paths(
        selected_pred, item_id, n_paths=N_PATHS, seed=SEED
    )
    paths_by_item[item_id] = item_paths
    ltd_by_item[item_id] = item_paths.sum(axis=0)
    inv_paths_by_item[item_id] = get_inventory_path(selected_sim, item_id)

if SELECTED_ITEM is None:
    paths = paths_by_item
    ltd_samples = ltd_by_item
    inv_path = inv_paths_by_item
    item_summary = pd.DataFrame([
        {
            "item_id": item_id,
            "ltd_mean": samples.mean(),
            f"ltd_q{int(TARGET_SERVICE_LEVEL * 100)}": samples.quantile(TARGET_SERVICE_LEVEL),
            "holding_cost": inv_paths_by_item[item_id]["holding_cost"].sum(),
            "stockout_cost": inv_paths_by_item[item_id]["stockout_cost"].sum(),
        }
        for item_id, samples in ltd_by_item.items()
    ]).sort_values("holding_cost", ascending=False).reset_index(drop=True)
else:
    only_item = item_ids[0]
    paths = paths_by_item[only_item]
    ltd_samples = ltd_by_item[only_item]
    inv_path = inv_paths_by_item[only_item]

print(f"Selected model: {SELECTED_MODEL}")
if SELECTED_ITEM is None:
    print(f"Demand paths generated for all {len(paths_by_item)} items")
else:
    print(f"Selected item: {item_ids[0]}")
    print(f"Demand paths: {paths.shape[0]} protection days x {paths.shape[1]} simulations")
    print(
        f"LTD mean: {ltd_samples.mean():.2f}; "
        f"q{TARGET_SERVICE_LEVEL * 100:.0f}: {ltd_samples.quantile(TARGET_SERVICE_LEVEL):.2f}"
    )

display(cost_df)
if SELECTED_ITEM is None:
    display(item_summary.head(20))
else:
    display(paths.head())
    display(inv_path.head())

cost_df


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.105088 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5752
[LightGBM] [Info] Number of data points in the train set: 425400, number of used features: 58
[LightGBM] [Info] Start training from score 0.371288
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.092392 seconds.
You can set `force_col_wise

,model,holding_cost,stockout_cost,total_cost
0,plain,15419.827918,1323.113006,16742.940925
1,hurdle,15601.706063,1169.244575,16770.950638
2,baseline:simple_moving_average,17440.962222,1567.000794,19007.963016
3,baseline:seasonal_moving_average,18147.627624,1536.017871,19683.645495
4,baseline:seasonal_naive,18580.050000,1604.857143,20184.907143


,item_id,ltd_mean,ltd_q83,holding_cost,stockout_cost
0,FOODS_3_635,165.708653,193.088810,887.428247,0.000000
1,FOODS_3_295,172.828589,195.669365,520.874203,0.000000
2,FOODS_3_099,205.149904,234.086755,504.520158,0.000000
3,FOODS_3_444,209.813441,237.375705,400.859066,32.860335
4,FOODS_2_197,200.185699,228.204324,391.383698,4.202792
5,FOODS_3_234,126.314505,153.516820,371.310606,3.437022
6,FOODS_3_580,51.603034,69.873838,329.532244,0.000000
7,FOODS_2_019,167.773733,193.717303,328.870006,3.019839
8,HOBBIES_1_032,53.759173,72.472927,249.329838,0.000000
9,FOODS_3_389,109.790356,121.724003,234.069028,0.000000


,model,holding_cost,stockout_cost,total_cost
0,plain,15419.827918,1323.113006,16742.940925
1,hurdle,15601.706063,1169.244575,16770.950638
2,baseline:simple_moving_average,17440.962222,1567.000794,19007.963016
3,baseline:seasonal_moving_average,18147.627624,1536.017871,19683.645495
4,baseline:seasonal_naive,18580.050000,1604.857143,20184.907143


In [ ]:
# Holdout test (adapted from src.backtest_engine.BacktestEngine.run_holdout_test)
# Uses calibration-window behavior (window_id==0 style): first 28 days of train slice used to build initial OOS history via InventoryPolicy.daily_simulation on that calibration eval.
# NOTE: keep this separate from run_prob_backtest; call run_holdout_prob_test(...) for a final holdout evaluation.

import re
from scripts.sku_diagnostics import per_sku_quantile_metrics

_QCOL = re.compile(r"q(\d{1,3})")


def _quantile_levels_nb(pred_df: pd.DataFrame) -> dict[str, float]:
    return {
        c: int(m.group(1)) / 100.0
        for c in pred_df.columns
        if (m := _QCOL.fullmatch(c)) and 0 < int(m.group(1)) < 100
    }


def run_holdout_prob_test(
    full_data: pd.DataFrame,
    test_holdout: pd.DataFrame,
    forecast_fn,
    feat_builder,
    base_backtest,          # BacktestEngine instance (for _baselines_metrics)
    training_year: int = 1,
    horizon: int = 28,
    lead_time: int = 4,
    review_period: int = 7,
    coverage_days: int = 4,
    policy: str = "quantile",
    safety_stock_type_override: str | None = None,
):
    \"\"\"Holdout test: train on last `training_year` years of full_data, evaluate on test_holdout.
    Mirrors BacktestEngine.run_holdout_test flow + calibration (window_id==0) for rmse-style OOS init inside InventoryPolicy
    by running sim on the calibration split first (updates oos_errors through daily_simulation).
    \"\"\"\"
    ss_policy = safety_stock_type_override or policy
    if ss_policy not in ("rmse", "quantile", "coverage"):
        raise ValueError(ss_policy)

    train_window_days = training_year * 365
    training_full = full_data[
        full_data["date"] >= (full_data["date"].max() - pd.Timedelta(days=train_window_days))
    ].copy()

    train_start = training_full["date"].min()
    train_end = training_full["date"].max()
    test_start = test_holdout["date"].min()
    test_end = test_holdout["date"].max()

    merged = pd.concat([training_full, test_holdout], axis=0, ignore_index=True)
    assert merged["date"].max() == test_end, "Merge error"

    merged_price = feat_builder.add_price_features(merged)
    train_price = merged_price[merged_price["date"].between(train_start, train_end)].copy()
    test_price = merged_price[merged_price["date"].between(test_start, test_end)].copy()

    # -- calibration split (like BacktestEngine window_id==0) --
    calibration_set = train_price.copy()
    cal_train_start = calibration_set["date"].min()
    cal_train_end = calibration_set["date"].max() - pd.Timedelta(days=horizon)
    cal_train, cal_eval = split_data(
        df=calibration_set,
        start_date=cal_train_start,
        end_date=cal_train_end,
        forecast_horizon=horizon,
    )

    # Build features for calibration
    profile_cal_tr = feat_builder.build_sku_demand_profile(cal_train)
    cal_train_fp = cal_train.merge(profile_cal_tr, on="item_id", how="left")
    cal_eval_fp = cal_eval.merge(profile_cal_tr, on="item_id", how="left")
    cal_train_fp, _ = feat_builder.build(df=cal_train_fp)

    base_preds_cal, _ = base_backtest._baselines_metrics(train_df=cal_train, eval_df=cal_eval)
    pred_cal = forecast_fn(cal_train_fp, cal_eval_fp, type="direct")
    if "sales_pred" not in pred_cal and "point" in pred_cal:
        pred_cal = pred_cal.rename(columns={"point": "sales_pred"})
    model_preds_cal = dict(base_preds_cal.items() if isinstance(base_preds_cal, dict) else base_preds_cal)
    model_preds_cal["ml-direct"] = pred_cal

    # Run calibration simulations to populate OOS error history inside InventoryPolicy instances
    cal_inventory = {}
    for name, preds in model_preds_cal.items():
        is_ml = name == "ml-direct"
        inv_pol_policy = ss_policy if (is_ml and ss_policy != "coverage") else ("coverage" if not is_ml else ss_policy)
        # For rmse you typically want rmse for ML; for baselines coverage is common; user controls via ss_policy
        if ss_policy == "rmse" and is_ml:
            inv_pol_policy = "rmse"
        elif ss_policy == "quantile" and is_ml:
            inv_pol_policy = "quantile"
        elif ss_policy == "coverage":
            inv_pol_policy = "coverage"
        else:
            inv_pol_policy = ss_policy if is_ml else "coverage"

        risk_period = lead_time + review_period
        initial_inventory = (
            cal_train.groupby("item_id", observed=True)["sales"].apply(lambda x: x.tail(28).mean())
        ) * risk_period
        preds_with_actuals = preds.merge(
            cal_eval[["item_id", "date", "sales"]].copy(), on=["item_id", "date"], how="left"
        )
        invp = InventoryPolicy(
            inventory=initial_inventory,
            forecasts=preds_with_actuals,
            lead_time=lead_time,
            review_period=review_period,
            coverage_days=coverage_days,
            daily_unit_holding_cost=0.2,
            stockout_cost=1.0,
            safety_stock_type=inv_pol_policy,
        )
        invp.daily_simulation(
            actual_sales=cal_eval,
            forecasted_demand=preds,
            model_name=name,
            forecast_type=name,
            store_id="CA_1",
        )
        cal_inventory[name] = invp  # keep OOS history seeded

    # -- main eval on test holdout --
    profile_te = feat_builder.build_sku_demand_profile(train_price)
    train_fp = train_price.merge(profile_te, on="item_id", how="left")
    test_fp = test_price.merge(profile_te, on="item_id", how="left")
    train_fp, _ = feat_builder.build(df=train_fp)

    base_preds_te, _ = base_backtest._baselines_metrics(train_df=train_price, eval_df=test_price)
    pred_te = forecast_fn(train_fp, test_fp, type="direct")
    if "sales_pred" not in pred_te and "point" in pred_te:
        pred_te = pred_te.rename(columns={"point": "sales_pred"})
    model_preds_te = dict(base_preds_te.items() if isinstance(base_preds_te, dict) else base_preds_te)
    model_preds_te["ml-direct"] = pred_te

    rows = []
    for name, preds in model_preds_te.items():
        is_ml = name == "ml-direct"
        if ss_policy == "rmse":
            inv_pol_policy = "rmse" if is_ml else "coverage"
        elif ss_policy == "quantile":
            inv_pol_policy = "quantile" if is_ml else "coverage"
        else:
            inv_pol_policy = "coverage"
        risk_period = lead_time + review_period
        initial_inventory = (
            train_price.groupby("item_id", observed=True)["sales"].apply(lambda x: x.tail(28).mean())
        ) * risk_period
        preds_with_actuals = preds.merge(
            test_price[["item_id", "date", "sales"]].copy(), on=["item_id", "date"], how="left"
        )
        # Seed RMSE with calibration OOS history if available
        oos_seed = None
        if inv_pol_policy == "rmse" and name in cal_inventory and cal_inventory[name].oos_errors:
            oos_seed = list(cal_inventory[name].oos_errors)
        invp = InventoryPolicy(
            inventory=initial_inventory,
            forecasts=preds_with_actuals,
            lead_time=lead_time,
            review_period=review_period,
            coverage_days=coverage_days,
            daily_unit_holding_cost=0.2,
            stockout_cost=1.0,
            safety_stock_type=inv_pol_policy,
            oos_error_list=oos_seed,
        )
        sim_df = invp.daily_simulation(
            actual_sales=test_price,
            forecasted_demand=preds,
            model_name=name,
            forecast_type=name,
            store_id="CA_1",
        )
        cost_summary = sim_df[["holding_cost", "stockout_cost"]].sum()
        cost_summary["total_cost"] = cost_summary.sum()
        rows.append({"model": name, "policy": inv_pol_policy, **cost_summary.to_dict(), "n_days": sim_df["date"].nunique()})

    results_df = pd.DataFrame(rows)
    return {
        "results": results_df,
        "train_start": train_start,
        "train_end": train_end,
        "test_start": test_start,
        "test_end": test_end,
        "cal_inventory": cal_inventory,
    }


In [ ]:
# Example: run holdout test (adapt paths/data as needed)
# You need a holdout dataframe, e.g. test_filtered_ca1.parquet or a split you saved.
holdout_path = BASE_DIR / 'data' / 'processed' / 'test_filtered_ca1.parquet'
if holdout_path.exists():
    test_holdout_df = pd.read_parquet(holdout_path)
    ho = run_holdout_prob_test(
        full_data=train_df,
        test_holdout=test_holdout_df,
        forecast_fn=plain_direct,
        feat_builder=feat_builder,
        base_backtest=backtest,
        training_year=1,
        horizon=28,
        lead_time=4,
        review_period=7,
        coverage_days=4,
        policy='quantile',  # or 'rmse' / 'coverage' for ML
    )
    display(ho['results'])
else:
    print('No holdout parquet found at', holdout_path)
